# iMAKS 전처리 + Ontology v1 실습
2026-10-05 개인 제출용. 원본 ZIP을 보존하고 변경은 파생 파일에만 반영합니다. **출력까지 실행한 노트북**입니다. 알고리즘 상세는 `prepare.py`, 설계 이유는 `ontology_v1.md`를 읽으세요.

## 1. 경로와 원본 확인
`submission` 폴더 또는 그 상위 프로젝트 폴더에서 실행할 수 있습니다.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / 'prepare.py').exists() else Path.cwd() / 'submission'
assert (ROOT / 'prepare.py').exists()
sys.path.insert(0, str(ROOT))
from prepare import prepare
summary = prepare()
display(pd.DataFrame([summary])[['rows','sensors','stations','time_start','time_end']])

,rows,sensors,stations,time_start,time_end
0,211200,22,9,2026-01-06 06:00:00,2026-01-09 13:59:30


## 2. 파일·컬럼 파악
센서 시계열과 KG는 `sensor_id ↔ Sensor.name`, `station_id ↔ Component.name`으로 연결합니다. 내부 KG 키는 `nodeId`입니다.

In [2]:
dictionary = pd.read_csv(ROOT / 'data_dictionary.csv')
display(dictionary.groupby('file', sort=False).agg(rows=('rows','first'), columns=('column','count')))
display(dictionary[dictionary.file.eq('sensors/timeseries_raw.csv')])

,rows,columns
file,,
kg_seed/edges.csv,341,4
kg_seed/ground_truth.csv,86,14
kg_seed/nodes.csv,115,35
kg_seed/nodes_factory.csv,24,5
sensors/timeseries_annotated.csv,211200,19
sensors/timeseries_raw.csv,211200,16


,file,rows,column,dtype,missing,unique,example,meaning
77,sensors/timeseries_raw.csv,211200,timestamp,str,0,9600,2026-01-06T06:00:00,측정 시각; 시간대 정보 없음
78,sensors/timeseries_raw.csv,211200,day,int64,0,5,1,원본 시뮬레이션 일차; 실제 timestamp와 교차 확인
79,sensors/timeseries_raw.csv,211200,shift,str,0,2,Morning,근무 교대
80,sensors/timeseries_raw.csv,211200,batch_id,str,0,10,BATCH-2026-01-M,배치 식별자
81,sensors/timeseries_raw.csv,211200,zone,str,0,6,Production Area,구역명
82,sensors/timeseries_raw.csv,211200,station_id,str,0,9,ST01_FILLING,스테이션 식별자; KG Component.name과 연결
83,sensors/timeseries_raw.csv,211200,sensor_id,str,0,22,ST01_FILLING_TMP,센서 식별자; KG Sensor.name과 연결
84,sensors/timeseries_raw.csv,211200,sensor_type,str,0,9,TMP,측정 유형 코드
85,sensors/timeseries_raw.csv,211200,value,float64,0,120738,22.5711,원래 단위의 측정값
86,sensors/timeseries_raw.csv,211200,unit,str,0,9,°C,측정 단위


## 3. 센서별 품질 점검
중복 키는 `(timestamp, sensor_id)`입니다. 같은 시각에 여러 센서가 측정한 것은 중복이 아닙니다.

In [3]:
raw = pd.read_csv(ROOT / 'data/raw/sensors/timeseries_raw.csv')
raw['parsed_time'] = pd.to_datetime(raw.timestamp, errors='coerce')
print('결측 셀:', raw.isna().sum().sum())
print('중복 키:', raw.duplicated(['timestamp','sensor_id']).sum())
display(pd.read_csv(ROOT / 'reports/sensor_checks.csv'))

결측 셀: 0
중복 키: 0


,sensor_id,rows,start,end,missing_cells,duplicate_keys,original_time_sorted,non_30_second_intervals,other_intervals_seconds,units,station_count,kg_sensor_match,min_value,max_value
0,CAF01_CAFETERIA_HUM,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},%RH,1,True,49.6898,60.5999
1,CAF01_CAFETERIA_TMP,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},°C,1,True,19.2314,27.5259
2,CHM01_CHEMICALSTORAGE_HUM,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},%RH,1,True,31.0488,38.4516
3,CHM01_CHEMICALSTORAGE_TMP,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},°C,1,True,13.8810,22.4654
4,RND01_RDLAB_HUM,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},%RH,1,True,46.1496,69.6137
5,RND01_RDLAB_TMP,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},°C,1,True,21.2608,22.7334
6,SRV01_SERVERROOM_HUM,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},%RH,1,True,41.3503,48.8183
7,SRV01_SERVERROOM_TMP,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},°C,1,True,19.7155,27.0437
8,ST01_FILLING_FLW,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},L/min,1,True,91.4678,125.1934
9,ST01_FILLING_PRS,9600,2026-01-06 06:00:00,2026-01-09 13:59:30,0,0,True,0,{},bar,1,True,3.6127,4.4345


## 4. 전처리 결과와 정답 분리
입력은 측정 시각·센서·스테이션·유형·값·단위만 허용합니다. 값 보정·평활화·정규화·보간은 하지 않았습니다. 정규화는 향후 학습 구간에서만 fit해야 합니다.

In [4]:
clean = pd.read_csv(ROOT / 'data/processed/timeseries_input.csv')
assert list(clean.columns) == ['timestamp','sensor_id','station_id','sensor_type','value','unit']
assert len(clean) == len(raw)
joined = clean.merge(raw[['timestamp','sensor_id','value']], on=['timestamp','sensor_id'], suffixes=('_out','_raw'), validate='one_to_one')
assert (joined.value_out == joined.value_raw).all()
print('측정값 및 행 수 보존 확인:', len(clean))
display(clean.head())

측정값 및 행 수 보존 확인: 211200


,timestamp,sensor_id,station_id,sensor_type,value,unit
0,2026-01-06T06:00:00,CAF01_CAFETERIA_HUM,CAF01_CAFETERIA,HUM,55.8419,%RH
1,2026-01-06T06:00:30,CAF01_CAFETERIA_HUM,CAF01_CAFETERIA,HUM,55.8659,%RH
2,2026-01-06T06:01:00,CAF01_CAFETERIA_HUM,CAF01_CAFETERIA,HUM,54.0637,%RH
3,2026-01-06T06:01:30,CAF01_CAFETERIA_HUM,CAF01_CAFETERIA,HUM,54.8028,%RH
4,2026-01-06T06:02:00,CAF01_CAFETERIA_HUM,CAF01_CAFETERIA,HUM,54.9306,%RH


## 5. KG 구조와 평가용 이벤트 확인
이 셀은 데이터 감사·평가 설계용입니다. 여기서 읽은 정답 이벤트를 탐지 특징이나 검색용 입력 KG에 넣지 않습니다.

In [5]:
display(pd.Series({k:summary[k] for k in ['node_count','edge_count','duplicate_node_ids','dangling_edges','metadata_mismatches','nodes_factory_conflicts','static_nodes','static_edges','production_events']}))
display(pd.read_csv(ROOT / 'reports/nodes_factory_conflicts.csv'))
events = pd.read_csv(ROOT / 'data/evaluation/events.csv')
display(events)
assert events.rows_in_interval.sum() == summary['event_interval_rows']
assert summary['anomaly_rows_outside_event_intervals'] == 0
assert not summary['event_label_mismatches']

node_count                 115
edge_count                 341
duplicate_node_ids           0
dangling_edges               0
metadata_mismatches          0
nodes_factory_conflicts      6
static_nodes                39
static_edges                72
production_events            9
dtype: int64

,nodeId,label_factory,name_factory,type,zone,label_seed,name_seed
0,N0018,Sensor,ST03_LABELLING_TMP,TMP,Production Area,Sensor,ST03_LABELLING_SPD
1,N0019,Sensor,ST03_LABELLING_SPD,SPD,Production Area,Sensor,ST03_LABELLING_TEN
2,N0020,Sensor,ST03_LABELLING_POS,POS,Production Area,Sensor,ST03_LABELLING_CNT
3,N0022,Sensor,ST04_PACKAGING_TMP,TMP,Production Area,Sensor,ST04_PACKAGING_VIB
4,N0023,Sensor,ST04_PACKAGING_SPD,SPD,Production Area,Sensor,ST04_PACKAGING_CUR
5,N0024,Sensor,ST04_PACKAGING_CNT,CNT,Production Area,Sensor,ST04_PACKAGING_SPD


,event_id,sensor_id,start,end_inclusive,anomaly_type,rows_in_interval,matching_label_rows,split_proposal,production_scope
0,GT-0001,ST02_SEALING_TMP,2026-01-06T08:30:00,2026-01-06T08:33:00,SPIKE,7,7,development,True
1,GT-0002,ST01_FILLING_PRS,2026-01-06T11:00:00,2026-01-06T11:45:00,DRIFT,91,91,development,True
2,GT-0003,ST04_PACKAGING_VIB,2026-01-07T07:00:00,2026-01-07T08:30:00,DRIFT,181,181,development,True
3,GT-0004,ST03_LABELLING_TEN,2026-01-07T09:30:00,2026-01-07T09:42:00,STUCK,25,25,development,True
4,GT-0005,SRV01_SERVERROOM_TMP,2026-01-07T12:00:00,2026-01-07T12:20:00,OUT_OF_RANGE,41,41,development,False
5,GT-0006,CAF01_CAFETERIA_TMP,2026-01-07T16:00:00,2026-01-07T16:30:00,OUT_OF_RANGE,61,61,development,False
6,GT-0007,ST01_FILLING_FLW,2026-01-08T06:30:00,2026-01-08T06:32:00,SPIKE,5,5,development,True
7,GT-0008,ST02_SEALING_CUR,2026-01-08T08:00:00,2026-01-08T10:00:00,DRIFT,241,241,development,True
8,GT-0009,ST04_PACKAGING_SPD,2026-01-08T09:30:00,2026-01-08T10:30:00,CORRELATED,121,121,development,True
9,GT-0010,ST03_LABELLING_CNT,2026-01-09T07:30:00,2026-01-09T07:55:00,OUT_OF_RANGE,51,51,test,True


## 6. 문서 추출과 출처
페이지별 텍스트, 레이아웃 텍스트, 표를 함께 보존합니다. 표 내부 줄바꿈과 페이지를 넘는 표는 LLM 추출 전에 추가 정리가 필요합니다.

In [6]:
docs = json.loads((ROOT / 'data/processed/documents.json').read_text(encoding='utf-8'))
display(pd.read_csv(ROOT / 'reports/document_checks.csv'))
sample = next(x for x in docs if x['document_id'] == 'SOP-003')
print(sample['source_file'], 'page', sample['page'])
display(pd.DataFrame(sample['tables'][0][1:], columns=sample['tables'][0][0]))

,file,page,chars,tables
0,datasheets/DS_Electrical_Sensors.pdf,1,818,1
1,datasheets/DS_Mechanical_Sensors.pdf,1,1114,4
2,datasheets/DS_Thermal_Sensors.pdf,1,976,2
3,rules/SOP_001_OperatingProcedures.pdf,1,2033,1
4,rules/SOP_001_OperatingProcedures.pdf,2,1435,0
5,rules/SOP_002_AlarmThresholds.pdf,1,1675,5
6,rules/SOP_002_AlarmThresholds.pdf,2,1660,6
7,rules/SOP_003_MaintenanceRules.pdf,1,1535,2
8,rules/SOP_004_PersonnelZoneAccess.pdf,1,1440,3
9,rules/SOP_004_PersonnelZoneAccess.pdf,2,155,0


rules/SOP_003_MaintenanceRules.pdf page 1


,Rule ID,Sens\nor,Station,Trigger,Action,Priority
0,MAINT-01,VIB,ST04_PACKAGING,VIB DRIFT >0.05 mm/s\nover 60 min,Inspect and lubricate\nbearing,HIGH
1,MAINT-02,CUR,ST02_SEALING,CUR DRIFT >1.0 A for\n30+ min,Inspect sealing jaw,HIGH
2,MAINT-03,TEN,ST03_LABELLING,TEN STUCK >10 samples,Replace sensor,MEDIUM
3,MAINT-04,PRS,ST01_FILLING,PRS <3.3 bar for >15 min,Inspect filter and pump,MEDIUM
4,MAINT-05,TMP,SRV01_SERVERRO\nOM,TMP >25°C for >5 min,Activate secondary\ncooling,CRITICAL
5,MAINT-06,TMP,WRH01_WAREHOU\nSE,TMP DRIFT >2°C over 2 h,Inspect compressor;\nnotify QC,CRITICAL
6,MAINT-07,TMP,CHM01_CHEMICALS\nTORAGE,TMP >20°C for >10 min,Emergency ventilation;\nnotify safety officer,CRITICAL
7,MAINT-08,HUM,RND01_RDLAB,HUM DRIFT >10%RH\nover 60 min,Activate dehumidifier;\ncheck HVAC drain,HIGH


## 7. SHACL 실습
정상 예제가 통과하고 잘못된 예제가 실패하는지 확인합니다. `sample_valid.ttl`의 이벤트는 연습용이며 실제 탐지 결과가 아닙니다.

In [7]:
from validate_graph import run
validation = pd.DataFrame(run())
display(validation)
assert validation.test_passed.all()

,case,expected_conforms,actual_conforms,violations,test_passed
0,valid,True,True,0,True
1,missing_sensor_id,False,False,1,True
2,missing_station,False,False,1,True
3,missing_start,False,False,1,True
4,missing_sensor_link,False,False,1,True
5,end_before_start,False,False,1,True
6,open_event,True,True,0,True
7,unknown_sensor,False,False,1,True
8,static_graph,True,True,0,True


## 8. 질문에 답하는 경로 확인
문서 점검 단계는 수동 예제입니다. 규칙의 발동 조건 충족이나 GraphRAG 성능을 평가한 결과가 아닙니다.

In [8]:
answers = json.loads((ROOT / 'reports/competency_queries.json').read_text(encoding='utf-8'))
for name, answer in answers.items():
    print(name, answer['rows'])

CQ1_sensor_station [['https://example.org/imaks/ST02_SEALING']]
CQ2_event_time [['2026-01-06T07:00:00', '2026-01-06T07:10:00']]
CQ3_inspection_evidence [['https://example.org/imaks/DEMO_STEP', 'Inspect sealing jaw', 'rules/SOP_003_MaintenanceRules.pdf', '1']]


## 9. 회의 준비
1. 제조 4개 스테이션으로 좁히면 이벤트는 9개입니다.
2. ZIP 실제 이벤트 ID·날짜가 소개 페이지 일부와 다릅니다. 파일 해시를 고정합니다.
3. 현재 SHACL은 구조를 검증하며 수치 임계값 및 문서 사실성 검증은 후속 단계입니다.
4. `README.md`와 `reports/findings.md`를 보고 본인이 설명할 수 있는지 확인하세요.